# Imports

In [1]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load data

In [2]:
maison_df = pd.read_csv('datasets_prepd/dvf_maison.csv')
print(maison_df.head())

    id_mutation date_mutation  numero_disposition nature_mutation  \
0  2021-1289503    2021-01-05                   1           Vente   
1  2021-1289508    2021-01-08                   1           Vente   
2  2021-1289509    2021-01-06                   1           Vente   
3  2021-1289510    2021-01-06                   1           Vente   
4  2021-1289511    2021-01-04                   1           Vente   

   valeur_fonciere  adresse_numero adresse_suffixe      adresse_nom_voie  \
0         352000.0           228.0             NaN       RUE DE L EGLISE   
1         235000.0             8.0             NaN          RUE MIRABEAU   
2         334800.0             7.0             NaN          AV ROSCOMMON   
3         460300.0            41.0               B       RUE NUMA GILLET   
4         225700.0            35.0             NaN  RUE DE LA REPUBLIQUE   

  adresse_code_voie  code_postal  ...  code_nature_culture_speciale  \
0              0220      77115.0  ...                    

# Features

In [3]:
features = [
    "longitude",
    "latitude",
    "code_postal",
    "surface_reelle_bati",
    "nombre_pieces_principales",
    "prix_m2_ref",
    "surface_terrain",
    "number_of_lots"
]

target = "valeur_fonciere_actualisee"

X = maison_df[features]
y = maison_df[target]

print("Number of features:", X.shape[1])
print(X.head())
print(y.head())

Number of features: 8
   longitude   latitude  code_postal  surface_reelle_bati  \
0   2.755044  48.527196      77115.0                120.0   
1   2.671257  48.768597      77330.0                 41.0   
2   2.693202  48.486025      77590.0                100.0   
3   2.738017  48.339558      77690.0                196.0   
4   2.585738  48.715644      77170.0                 71.0   

   nombre_pieces_principales  prix_m2_ref  surface_terrain  number_of_lots  
0                        3.0  3151.092187            331.0             NaN  
1                        2.0  5514.166105            175.0             NaN  
2                        4.0  4696.072809            337.0             NaN  
3                        5.0  3641.879093            800.0             NaN  
4                        3.0  6270.848107             55.0             NaN  
0    352000.0
1    235000.0
2    334800.0
3    460300.0
4    225700.0
Name: valeur_fonciere_actualisee, dtype: float64


# Make sets : trail and test and validate

In [4]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (33953, 8)
Validation: (7276, 8)
Test: (7276, 8)


# Ssave

In [5]:
os.makedirs("data_maison", exist_ok=True)

train_df = X_train.copy()
val_df = X_val.copy()
test_df = X_test.copy()

train_df[target] = y_train
val_df[target] = y_val
test_df[target] = y_test

train_df.to_csv("data_maison/maison_train.csv", index=False)
val_df.to_csv("data_maison/maison_val.csv", index=False)
test_df.to_csv("data_maison/maison_test.csv", index=False)

print("Datasets saved.")

Datasets saved.


Train model

In [6]:
model_maison = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

model_maison.fit(X_train, y_train)

print("Model trained.")

Model trained.


Evaluation

In [7]:
y_val_pred = model_maison.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
mae = mean_absolute_error(y_val, y_val_pred)
r2 = r2_score(y_val, y_val_pred)

print("Validation RMSE:", rmse)
print("Validation MAE:", mae)
print("Validation R2:", r2)

Validation RMSE: 84191.540649673
Validation MAE: 58121.85219185745
Validation R2: 0.6376444866232133


In [8]:
df_eval = pd.DataFrame({
    "prix_reel": y_val,
    "prix_predit": y_val_pred
})

df_eval["erreur_pct"] = abs(df_eval["prix_reel"] - df_eval["prix_predit"]) / df_eval["prix_reel"] * 100

print(df_eval.head())
print("Erreur moyenne (%) :", df_eval["erreur_pct"].mean())

       prix_reel    prix_predit  erreur_pct
45168   459375.0  669942.399225   45.837801
17026   341600.0  260015.692204   23.882994
30940   492200.0  543327.720756   10.387591
33373   135300.0  243846.902182   80.226831
35217   116611.0  169272.044594   45.159586
Erreur moyenne (%) : 20.74344133312875


# Test

In [9]:
y_train_pred = model_maison.predict(X_train)

rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
mae = mean_absolute_error(y_train, y_train_pred)
r2 = r2_score(y_train, y_train_pred)

print("Train RMSE:", rmse)
print("Train MAE:", mae)
print("Train R2:", r2)

Train RMSE: 38272.0533810418
Train MAE: 27216.68921229957
Train R2: 0.9267402855389615


Save model

In [10]:
os.makedirs("models", exist_ok=True)

joblib.dump(model_maison, "models/maison_random_forest_model.pkl")

print("Model saved.")

Model saved.


In [11]:
print("Number of features expected:", model_maison.n_features_in_)
print("Feature names:", features)

Number of features expected: 8
Feature names: ['longitude', 'latitude', 'code_postal', 'surface_reelle_bati', 'nombre_pieces_principales', 'prix_m2_ref', 'surface_terrain', 'number_of_lots']
